In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from IPython.display import display

BASE = (
    Path("expirement_v2")
    if Path("expirement_v2/data").is_dir()
    else Path(".")
)

model_data = xr.open_dataset(
    BASE / "data/processed/malta_forecast_prepared.nc"
)

feature_names = json.loads(
    model_data.attrs["feature_names_json"]
)
horizons = tuple(
    int(h) for h in model_data.attrs["forecast_horizons_days"]
)

print(f"Predictors: {len(feature_names)}")
print(f"Forecast horizons: {horizons}")
print(f"Issue time: {model_data.attrs['forecast_issue_time_utc']} UTC")

display(model_data)

Predictors: 20
Forecast horizons: (1, 4, 7)
Issue time: 20:30 UTC


<xarray.Dataset> Size: 1GB
Dimensions:              (time: 1461, latitude: 133, longitude: 101)
Coordinates:
  * time                 (time) datetime64[ns] 12kB 2022-01-01 ... 2025-12-31
  * latitude             (latitude) float32 532B 35.55 35.56 ... 36.88 36.89
  * longitude            (longitude) float32 404B 13.91 13.92 ... 15.18 15.19
Data variables: (12/27)
    season_sin           (time) float32 6kB ...
    season_cos           (time) float32 6kB ...
    chl_latest           (time, latitude, longitude) float32 79MB ...
    chl_age_days         (time, latitude, longitude) float32 79MB ...
    sst_c                (time, latitude, longitude) float32 79MB ...
    sst_observed         (time, latitude, longitude) uint8 20MB ...
    ...                   ...
    chl_target_4d        (time, latitude, longitude) float32 79MB ...
    chl_target_7d        (time, latitude, longitude) float32 79MB ...
    model_mask           (latitude, longitude) uint8 13kB ...
    split_train          (time) uint8 1kB ...
    split_validation     (time) uint8 1kB ...
    split_test           (time) uint8 1kB ...
Attributes:
    time_meaning:              Forecast issue date
    forecast_issue_time_utc:   20:30
    feature_names_json:        ["season_sin", "season_cos", "chl_latest", "ch...
    forecast_horizons_days:    [1 4 7]
    availability_basis:        Nominal NRT delivery schedules applied to MY o...
    target_source:             Observed L3 chlorophyll
    training_period:           2022–2023
    validation_period:         2024
    test_period:               2025
    source_observations_file:  malta_daily_observations.nc

In [2]:
validation = model_data.isel(
    time=np.flatnonzero(model_data["split_validation"].values == 1)
)

prediction = validation["chl_latest"].transpose(
    "time", "latitude", "longitude"
).values

rows = []

for horizon in horizons:
    observed = (
        validation[f"chl_target_{horizon}d"]
        .where(validation["model_mask"] == 1)
        .transpose("time", "latitude", "longitude")
        .values
    )

    valid_target = np.isfinite(observed) & (observed > 0)
    scored = valid_target & np.isfinite(prediction) & (prediction > 0)

    actual = observed[scored].astype("float64")
    predicted = prediction[scored].astype("float64")
    error = predicted - actual

    rows.append({
        "Horizon": f"+{horizon}d",
        "Scored pixel-days": len(actual),
        "Scored targets (%)": 100 * scored.sum() / valid_target.sum(),
        "MAE (mg/m³)": np.abs(error).mean(),
        "RMSE (mg/m³)": np.sqrt(np.mean(error ** 2)),
        "RMSE (log10)": np.sqrt(
            np.mean((np.log10(predicted) - np.log10(actual)) ** 2)
        ),
        "Median relative error (%)":
            np.median(100 * np.abs(error) / actual),
    })

persistence_validation = pd.DataFrame(rows).set_index("Horizon")

display(persistence_validation.style.format({
    "Scored pixel-days": "{:,.0f}",
    "Scored targets (%)": "{:.2f}",
    "MAE (mg/m³)": "{:.4f}",
    "RMSE (mg/m³)": "{:.4f}",
    "RMSE (log10)": "{:.4f}",
    "Median relative error (%)": "{:.1f}",
}))

,Scored pixel-days,Scored targets (%),MAE (mg/m³),RMSE (mg/m³),RMSE (log10),Median relative error (%)
Horizon,,,,,,
+1d,"2,507,558",100.00,0.0172,0.0548,0.1205,13.4
+4d,"2,514,892",100.00,0.0202,0.0650,0.1383,15.0
+7d,"2,541,638",100.00,0.0225,0.0696,0.1546,16.4


In [3]:
with xr.open_dataset(
    BASE / "data/processed/malta_daily_observations.nc"
) as observations:
    training_chl = (
        observations["chl"]
        .sel(time=slice("2022-01-01", "2023-12-31"))
        .where(model_data["model_mask"] == 1)
        .load()
    )

monthly_climatology = training_chl.groupby("time.month").median(
    dim="time", skipna=True
)

# Use the pixel's overall training median when a month has no observations.
pixel_training_median = training_chl.median(dim="time", skipna=True)
monthly_climatology = monthly_climatology.fillna(pixel_training_median)

display(monthly_climatology)

<xarray.DataArray 'chl' (month: 12, latitude: 133, longitude: 101)> Size: 645kB
array([[[0.0958578 , 0.10406856, 0.09988315, ..., 0.14176762,
         0.14268045, 0.13616341],
        [0.1018592 , 0.10647546, 0.10155682, ..., 0.13917924,
         0.14229059, 0.13918045],
        [0.11406975, 0.11172222, 0.0967267 , ..., 0.13619106,
         0.13462996, 0.14428571],
        ...,
        [0.19279042, 0.18387744, 0.19246769, ..., 0.18609414,
         0.17196175, 0.16785169],
        [0.18737862, 0.19116518, 0.19692028, ..., 0.16199782,
         0.15833443, 0.1622036 ],
        [0.19538975, 0.19222456, 0.1966607 , ..., 0.16384678,
         0.16195795, 0.16360867]],

       [[0.13289014, 0.13162504, 0.12621877, ..., 0.16264072,
         0.16166683, 0.17113131],
        [0.13576491, 0.12879737, 0.12618414, ..., 0.16340217,
         0.16686764, 0.17337175],
        [0.1388583 , 0.12655719, 0.12909165, ..., 0.16415535,
         0.15313302, 0.17409477],
...
        [0.06514788, 0.06521931, 0.06718501, ..., 0.12282019,
         0.10871682, 0.09771326],
        [0.06627249, 0.06615706, 0.06757853, ..., 0.12942964,
         0.11639345, 0.10371453],
        [0.06705459, 0.06642323, 0.06899204, ..., 0.13869815,
         0.11279738, 0.10779043]],

       [[0.06748877, 0.0665075 , 0.06542677, ..., 0.09174515,
         0.10162376, 0.09848123],
        [0.06838378, 0.06598395, 0.06583403, ..., 0.09271151,
         0.09624727, 0.09415452],
        [0.0693104 , 0.06365369, 0.06394923, ..., 0.09764387,
         0.0960712 , 0.09507523],
        ...,
        [0.12790644, 0.11712609, 0.12624548, ..., 0.12542327,
         0.12715527, 0.1339221 ],
        [0.12875359, 0.12538636, 0.12803951, ..., 0.12680319,
         0.12667243, 0.12899186],
        [0.12667671, 0.12353826, 0.12860806, ..., 0.12061462,
         0.12820727, 0.13136758]]], shape=(12, 133, 101), dtype=float32)
Coordinates:
  * month      (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
  * latitude   (latitude) float32 532B 35.55 35.56 35.57 ... 36.87 36.88 36.89
  * longitude  (longitude) float32 404B 13.91 13.92 13.93 ... 15.17 15.18 15.19

In [4]:
def chl_scores(actual, predicted):
    actual = actual.astype("float64")
    predicted = predicted.astype("float64")
    error = predicted - actual

    return {
        "Scored pixel-days": len(actual),
        "MAE (mg/m³)": np.abs(error).mean(),
        "RMSE (mg/m³)": np.sqrt(np.mean(error ** 2)),
        "RMSE (log10)": np.sqrt(
            np.mean((np.log10(predicted) - np.log10(actual)) ** 2)
        ),
        "Median relative error (%)":
            np.median(100 * np.abs(error) / actual),
    }


persistence = validation["chl_latest"].transpose(
    "time", "latitude", "longitude"
).values

rows = []

for horizon in horizons:
    actual = (
        validation[f"chl_target_{horizon}d"]
        .where(validation["model_mask"] == 1)
        .transpose("time", "latitude", "longitude")
        .values
    )

    target_month = (
        validation.time + np.timedelta64(horizon, "D")
    ).dt.month

    seasonal = monthly_climatology.sel(month=target_month).transpose(
        "time", "latitude", "longitude"
    ).values

    common = (
        np.isfinite(actual) & (actual > 0)
        & np.isfinite(persistence) & (persistence > 0)
        & np.isfinite(seasonal) & (seasonal > 0)
    )

    for name, predicted in [
        ("Persistence", persistence),
        ("Seasonal median", seasonal),
    ]:
        rows.append({
            "Horizon": f"+{horizon}d",
            "Baseline": name,
            **chl_scores(actual[common], predicted[common]),
        })

baseline_validation = pd.DataFrame(rows).set_index(
    ["Horizon", "Baseline"]
)

display(baseline_validation.style.format({
    "Scored pixel-days": "{:,.0f}",
    "MAE (mg/m³)": "{:.4f}",
    "RMSE (mg/m³)": "{:.4f}",
    "RMSE (log10)": "{:.4f}",
    "Median relative error (%)": "{:.1f}",
}))

In [5]:
HORIZON = 1

train = model_data.isel(
    time=np.flatnonzero(model_data["split_train"].values == 1)
)

target = train[f"chl_target_{HORIZON}d"].transpose(
    "time", "latitude", "longitude"
)

valid_target = (
    np.isfinite(target)
    & (target > 0)
    & (train["model_mask"] == 1)
)

time_idx, latitude_idx, longitude_idx = np.nonzero(valid_target.values)

sample_indexers = {
    "time": xr.DataArray(time_idx, dims="sample"),
    "latitude": xr.DataArray(latitude_idx, dims="sample"),
    "longitude": xr.DataArray(longitude_idx, dims="sample"),
}

sample_features = train[feature_names].isel(sample_indexers)

X_train = (
    sample_features
    .drop_vars(["time", "latitude", "longitude"])
    .to_dataframe()[feature_names]
    .astype("float32")
)

y_train = np.log10(
    target.values[time_idx, latitude_idx, longitude_idx]
).astype("float32")

train_dates = train.time.values[time_idx]

print(f"Training table: {X_train.shape[0]:,} rows × {X_train.shape[1]} features")
print(f"Feature-table memory: {X_train.memory_usage().sum() / 1024**2:,.1f} MB")
display(X_train.head())

Training table: 5,374,856 rows × 20 features
Feature-table memory: 410.1 MB


,season_sin,season_cos,chl_latest,chl_age_days,sst_c,sst_observed,wind_u_asc,wind_v_asc,wind_speed_asc,wind_asc_observed,wind_u_des,wind_v_des,wind_speed_des,wind_des_observed,wind_age_hours_asc,wind_age_hours_des,chl_mean_7d,chl_observations_7d,latitude_deg,longitude_deg
sample,,,,,,,,,,,,,,,,,,,,
0,0.0,1.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,35.554432,13.907421
1,0.0,1.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,35.554432,13.920269
2,0.0,1.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,35.554432,13.933117
3,0.0,1.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,35.554432,13.945964
4,0.0,1.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,35.554432,13.958812


In [6]:
from sklearn.ensemble import HistGradientBoostingRegressor
from time import perf_counter

model_1d = HistGradientBoostingRegressor(
    loss="squared_error",
    learning_rate=0.05,
    max_iter=150,
    max_leaf_nodes=31,
    min_samples_leaf=100,
    l2_regularization=1.0,
    early_stopping=False,
    random_state=42,
    verbose=1,
)

started = perf_counter()
model_1d.fit(X_train, y_train)

print(f"Training completed in {(perf_counter() - started) / 60:.1f} minutes")
print(f"Boosting iterations: {model_1d.n_iter_}")

Binning 0.860 GB of training data: 0.824 s
Fitting gradient boosted rounds:
Fit 150 trees in 21.746 s, (4650 total leaves)
Time spent computing histograms: 13.256s
Time spent finding best splits:  0.204s
Time spent applying splits:      2.591s
Time spent predicting:           0.716s
Training completed in 0.4 minutes
Boosting iterations: 150


In [7]:
target_val = validation["chl_target_1d"].transpose(
    "time", "latitude", "longitude"
)

valid_val = (
    np.isfinite(target_val)
    & (target_val > 0)
    & (validation["model_mask"] == 1)
)

val_time_idx, val_lat_idx, val_lon_idx = np.nonzero(valid_val.values)

validation_indexers = {
    "time": xr.DataArray(val_time_idx, dims="sample"),
    "latitude": xr.DataArray(val_lat_idx, dims="sample"),
    "longitude": xr.DataArray(val_lon_idx, dims="sample"),
}

X_val = (
    validation[feature_names]
    .isel(validation_indexers)
    .drop_vars(["time", "latitude", "longitude"])
    .to_dataframe()[feature_names]
    .astype("float32")
)

y_val = target_val.values[val_time_idx, val_lat_idx, val_lon_idx]

prediction_1d = 10.0 ** model_1d.predict(X_val)

comparison_1d = baseline_validation.loc["+1d"].copy()

# Confirm that all models are scored on the full valid-target population.
assert (comparison_1d["Scored pixel-days"] == len(y_val)).all()

comparison_1d.loc["Gradient boosting"] = pd.Series(
    chl_scores(y_val, prediction_1d)
)

display(comparison_1d.style.format({
    "Scored pixel-days": "{:,.0f}",
    "MAE (mg/m³)": "{:.4f}",
    "RMSE (mg/m³)": "{:.4f}",
    "RMSE (log10)": "{:.4f}",
    "Median relative error (%)": "{:.1f}",
}))

,Scored pixel-days,MAE (mg/m³),RMSE (mg/m³),RMSE (log10),Median relative error (%)
Baseline,,,,,
Persistence,"2,507,558",0.0172,0.0548,0.1205,13.4
Seasonal median,"2,507,558",0.0221,0.0541,0.1432,18.1
Gradient boosting,"2,507,558",0.0153,0.0472,0.1057,12.3


In [8]:
target_dates = validation.time + np.timedelta64(1, "D")
target_seasons = target_dates.dt.season.values[val_time_idx]

seasonal_grid = monthly_climatology.sel(
    month=target_dates.dt.month
).transpose("time", "latitude", "longitude")

predictions = {
    "Persistence": X_val["chl_latest"].to_numpy(),
    "Seasonal median": seasonal_grid.values[
        val_time_idx, val_lat_idx, val_lon_idx
    ],
    "Gradient boosting": prediction_1d,
}

rows = []

for season, name in [
    ("DJF", "Winter"),
    ("MAM", "Spring"),
    ("JJA", "Summer"),
    ("SON", "Autumn"),
]:
    selected = target_seasons == season

    for method, predicted in predictions.items():
        rows.append({
            "Season": name,
            "Method": method,
            **chl_scores(y_val[selected], predicted[selected]),
        })

seasonal_comparison_1d = pd.DataFrame(rows).set_index(
    ["Season", "Method"]
)

display(seasonal_comparison_1d.style.format({
    "Scored pixel-days": "{:,.0f}",
    "MAE (mg/m³)": "{:.4f}",
    "RMSE (mg/m³)": "{:.4f}",
    "RMSE (log10)": "{:.4f}",
    "Median relative error (%)": "{:.1f}",
}))

In [9]:
age_groups = pd.cut(
    X_val["chl_age_days"].to_numpy(),
    bins=[0, 1, 2, 3, 7, np.inf],
    labels=["1", "2", "3", "4–7", "8+"],
)

rows = []

for age_label in ["1", "2", "3", "4–7", "8+"]:
    selected = age_groups == age_label

    if not selected.any():
        continue

    for method in ["Persistence", "Gradient boosting"]:
        rows.append({
            "CHL age (days)": age_label,
            "Method": method,
            **chl_scores(
                y_val[selected],
                predictions[method][selected],
            ),
        })

age_comparison_1d = pd.DataFrame(rows).set_index(
    ["CHL age (days)", "Method"]
)

display(age_comparison_1d.style.format({
    "Scored pixel-days": "{:,.0f}",
    "MAE (mg/m³)": "{:.4f}",
    "RMSE (mg/m³)": "{:.4f}",
    "RMSE (log10)": "{:.4f}",
    "Median relative error (%)": "{:.1f}",
}))

In [10]:
from sklearn.base import clone

core_features = [
    "season_sin",
    "season_cos",
    "chl_latest",
    "chl_age_days",
    "chl_mean_7d",
    "chl_observations_7d",
    "latitude_deg",
    "longitude_deg",
]

model_core_1d = clone(model_1d).set_params(verbose=0)
model_core_1d.fit(X_train[core_features], y_train)

prediction_core_1d = 10.0 ** model_core_1d.predict(
    X_val[core_features]
)

feature_comparison_1d = comparison_1d.rename(
    index={"Gradient boosting": "Full model: 20 features"}
).copy()

feature_comparison_1d.loc["CHL + season + location: 8 features"] = (
    pd.Series(chl_scores(y_val, prediction_core_1d))
)

display(feature_comparison_1d.style.format({
    "Scored pixel-days": "{:,.0f}",
    "MAE (mg/m³)": "{:.4f}",
    "RMSE (mg/m³)": "{:.4f}",
    "RMSE (log10)": "{:.4f}",
    "Median relative error (%)": "{:.1f}",
}))

,Scored pixel-days,MAE (mg/m³),RMSE (mg/m³),RMSE (log10),Median relative error (%)
Baseline,,,,,
Persistence,"2,507,558",0.0172,0.0548,0.1205,13.4
Seasonal median,"2,507,558",0.0221,0.0541,0.1432,18.1
Full model: 20 features,"2,507,558",0.0153,0.0472,0.1057,12.3
CHL + season + location: 8 features,"2,507,558",0.0160,0.0477,0.1101,12.8


In [11]:
sst_features = core_features + ["sst_c", "sst_observed"]

model_sst_1d = clone(model_1d).set_params(verbose=0)
model_sst_1d.fit(X_train[sst_features], y_train)

prediction_sst_1d = 10.0 ** model_sst_1d.predict(
    X_val[sst_features]
)

feature_comparison_1d.loc["Core + SST: 10 features"] = pd.Series(
    chl_scores(y_val, prediction_sst_1d)
)

display(feature_comparison_1d.style.format({
    "Scored pixel-days": "{:,.0f}",
    "MAE (mg/m³)": "{:.4f}",
    "RMSE (mg/m³)": "{:.4f}",
    "RMSE (log10)": "{:.4f}",
    "Median relative error (%)": "{:.1f}",
}))

,Scored pixel-days,MAE (mg/m³),RMSE (mg/m³),RMSE (log10),Median relative error (%)
Baseline,,,,,
Persistence,"2,507,558",0.0172,0.0548,0.1205,13.4
Seasonal median,"2,507,558",0.0221,0.0541,0.1432,18.1
Full model: 20 features,"2,507,558",0.0153,0.0472,0.1057,12.3
CHL + season + location: 8 features,"2,507,558",0.0160,0.0477,0.1101,12.8
Core + SST: 10 features,"2,507,558",0.0154,0.0474,0.1069,12.5


In [12]:
wind_features = core_features + [
    name for name in feature_names
    if name.startswith("wind_")
]

model_wind_1d = clone(model_1d).set_params(verbose=0)
model_wind_1d.fit(X_train[wind_features], y_train)

prediction_wind_1d = 10.0 ** model_wind_1d.predict(
    X_val[wind_features]
)

feature_comparison_1d.loc[
    f"Core + wind: {len(wind_features)} features"
] = pd.Series(chl_scores(y_val, prediction_wind_1d))

display(feature_comparison_1d.style.format({
    "Scored pixel-days": "{:,.0f}",
    "MAE (mg/m³)": "{:.4f}",
    "RMSE (mg/m³)": "{:.4f}",
    "RMSE (log10)": "{:.4f}",
    "Median relative error (%)": "{:.1f}",
}))

,Scored pixel-days,MAE (mg/m³),RMSE (mg/m³),RMSE (log10),Median relative error (%)
Baseline,,,,,
Persistence,"2,507,558",0.0172,0.0548,0.1205,13.4
Seasonal median,"2,507,558",0.0221,0.0541,0.1432,18.1
Full model: 20 features,"2,507,558",0.0153,0.0472,0.1057,12.3
CHL + season + location: 8 features,"2,507,558",0.0160,0.0477,0.1101,12.8
Core + SST: 10 features,"2,507,558",0.0154,0.0474,0.1069,12.5
Core + wind: 18 features,"2,507,558",0.0159,0.0474,0.1087,12.6


In [13]:
def make_samples(dataset, horizon):
    target = dataset[f"chl_target_{horizon}d"].transpose(
        "time", "latitude", "longitude"
    )

    valid = (
        np.isfinite(target)
        & (target > 0)
        & (dataset["model_mask"] == 1)
    )

    positions = np.nonzero(valid.values)

    indexers = {
        dimension: xr.DataArray(position, dims="sample")
        for dimension, position in zip(target.dims, positions)
    }

    X = (
        dataset[feature_names]
        .isel(indexers)
        .drop_vars(["time", "latitude", "longitude"])
        .to_dataframe()[feature_names]
        .astype("float32")
    )

    observed_chl = target.values[positions]

    return X, observed_chl, positions


X_train_4d, chl_train_4d, train_positions_4d = make_samples(train, 4)

print(f"+4-day training table: {X_train_4d.shape[0]:,} rows")
display(X_train_4d.head())

+4-day training table: 5,365,149 rows


,season_sin,season_cos,chl_latest,chl_age_days,sst_c,sst_observed,wind_u_asc,wind_v_asc,wind_speed_asc,wind_asc_observed,wind_u_des,wind_v_des,wind_speed_des,wind_des_observed,wind_age_hours_asc,wind_age_hours_des,chl_mean_7d,chl_observations_7d,latitude_deg,longitude_deg
sample,,,,,,,,,,,,,,,,,,,,
0,0.0,1.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,35.554432,13.907421
1,0.0,1.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,35.554432,13.920269
2,0.0,1.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,35.554432,13.933117
3,0.0,1.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,35.554432,13.945964
4,0.0,1.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,35.554432,13.958812


In [14]:
model_4d = clone(model_1d)

started = perf_counter()

model_4d.fit(
    X_train_4d,
    np.log10(chl_train_4d).astype("float32"),
)

print(f"+4-day training completed in {(perf_counter() - started) / 60:.1f} minutes")
print(f"Boosting iterations: {model_4d.n_iter_}")

Binning 0.858 GB of training data: 1.190 s
Fitting gradient boosted rounds:
Fit 150 trees in 23.014 s, (4650 total leaves)
Time spent computing histograms: 15.188s
Time spent finding best splits:  0.233s
Time spent applying splits:      2.943s
Time spent predicting:           0.766s
+4-day training completed in 0.4 minutes
Boosting iterations: 150


In [15]:
X_val_4d, chl_val_4d, val_positions_4d = make_samples(validation, 4)

prediction_4d = 10.0 ** model_4d.predict(X_val_4d)

comparison_4d = baseline_validation.loc["+4d"].copy()

assert (
    comparison_4d["Scored pixel-days"] == len(chl_val_4d)
).all()

comparison_4d.loc["Gradient boosting"] = pd.Series(
    chl_scores(chl_val_4d, prediction_4d)
)

display(comparison_4d.style.format({
    "Scored pixel-days": "{:,.0f}",
    "MAE (mg/m³)": "{:.4f}",
    "RMSE (mg/m³)": "{:.4f}",
    "RMSE (log10)": "{:.4f}",
    "Median relative error (%)": "{:.1f}",
}))

,Scored pixel-days,MAE (mg/m³),RMSE (mg/m³),RMSE (log10),Median relative error (%)
Baseline,,,,,
Persistence,"2,514,892",0.0202,0.0650,0.1383,15.0
Seasonal median,"2,514,892",0.0221,0.0540,0.1433,18.1
Gradient boosting,"2,514,892",0.0181,0.0512,0.1240,14.9


In [16]:
X_train_7d, chl_train_7d, train_positions_7d = make_samples(train, 7)

model_7d = clone(model_1d)

started = perf_counter()

model_7d.fit(
    X_train_7d,
    np.log10(chl_train_7d).astype("float32"),
)

print(f"Training rows: {len(chl_train_7d):,}")
print(f"+7-day training completed in {(perf_counter() - started) / 60:.1f} minutes")
print(f"Boosting iterations: {model_7d.n_iter_}")

Binning 0.858 GB of training data: 1.457 s
Fitting gradient boosted rounds:
Fit 150 trees in 26.234 s, (4650 total leaves)
Time spent computing histograms: 17.303s
Time spent finding best splits:  0.276s
Time spent applying splits:      3.265s
Time spent predicting:           0.856s
Training rows: 5,359,885
+7-day training completed in 0.4 minutes
Boosting iterations: 150


In [17]:
X_val_7d, chl_val_7d, val_positions_7d = make_samples(validation, 7)

prediction_7d = 10.0 ** model_7d.predict(X_val_7d)

comparison_7d = baseline_validation.loc["+7d"].copy()

assert (
    comparison_7d["Scored pixel-days"] == len(chl_val_7d)
).all()

comparison_7d.loc["Gradient boosting"] = pd.Series(
    chl_scores(chl_val_7d, prediction_7d)
)

display(comparison_7d.style.format({
    "Scored pixel-days": "{:,.0f}",
    "MAE (mg/m³)": "{:.4f}",
    "RMSE (mg/m³)": "{:.4f}",
    "RMSE (log10)": "{:.4f}",
    "Median relative error (%)": "{:.1f}",
}))

,Scored pixel-days,MAE (mg/m³),RMSE (mg/m³),RMSE (log10),Median relative error (%)
Baseline,,,,,
Persistence,"2,541,638",0.0225,0.0696,0.1546,16.4
Seasonal median,"2,541,638",0.0223,0.0541,0.1437,18.2
Gradient boosting,"2,541,638",0.0190,0.0520,0.1303,15.5


In [18]:
model_sst_4d = clone(model_4d).set_params(verbose=0)

model_sst_4d.fit(
    X_train_4d[sst_features],
    np.log10(chl_train_4d).astype("float32"),
)

prediction_sst_4d = 10.0 ** model_sst_4d.predict(
    X_val_4d[sst_features]
)

feature_comparison_4d = comparison_4d.rename(
    index={"Gradient boosting": "Full model: 20 features"}
).copy()

feature_comparison_4d.loc["Core + SST: 10 features"] = pd.Series(
    chl_scores(chl_val_4d, prediction_sst_4d)
)

display(feature_comparison_4d.style.format({
    "Scored pixel-days": "{:,.0f}",
    "MAE (mg/m³)": "{:.4f}",
    "RMSE (mg/m³)": "{:.4f}",
    "RMSE (log10)": "{:.4f}",
    "Median relative error (%)": "{:.1f}",
}))

,Scored pixel-days,MAE (mg/m³),RMSE (mg/m³),RMSE (log10),Median relative error (%)
Baseline,,,,,
Persistence,"2,514,892",0.0202,0.0650,0.1383,15.0
Seasonal median,"2,514,892",0.0221,0.0540,0.1433,18.1
Full model: 20 features,"2,514,892",0.0181,0.0512,0.1240,14.9
Core + SST: 10 features,"2,514,892",0.0181,0.0514,0.1240,14.8


In [19]:
model_sst_7d = clone(model_7d).set_params(verbose=0)

model_sst_7d.fit(
    X_train_7d[sst_features],
    np.log10(chl_train_7d).astype("float32"),
)

prediction_sst_7d = 10.0 ** model_sst_7d.predict(
    X_val_7d[sst_features]
)

feature_comparison_7d = comparison_7d.rename(
    index={"Gradient boosting": "Full model: 20 features"}
).copy()

feature_comparison_7d.loc["Core + SST: 10 features"] = pd.Series(
    chl_scores(chl_val_7d, prediction_sst_7d)
)

display(feature_comparison_7d.style.format({
    "Scored pixel-days": "{:,.0f}",
    "MAE (mg/m³)": "{:.4f}",
    "RMSE (mg/m³)": "{:.4f}",
    "RMSE (log10)": "{:.4f}",
    "Median relative error (%)": "{:.1f}",
}))

,Scored pixel-days,MAE (mg/m³),RMSE (mg/m³),RMSE (log10),Median relative error (%)
Baseline,,,,,
Persistence,"2,541,638",0.0225,0.0696,0.1546,16.4
Seasonal median,"2,541,638",0.0223,0.0541,0.1437,18.2
Full model: 20 features,"2,541,638",0.0190,0.0520,0.1303,15.5
Core + SST: 10 features,"2,541,638",0.0189,0.0517,0.1297,15.5


In [20]:
FINAL_FEATURES = sst_features.copy()
FINAL_MODEL_PARAMS = model_sst_1d.get_params()
FINAL_MODEL_PARAMS["verbose"] = 0

latest_target_dates = (
    model_data.time + np.timedelta64(max(horizons), "D")
)

final_fit_mask = (
    (model_data.time >= np.datetime64("2022-01-01"))
    & (latest_target_dates <= np.datetime64("2024-12-31"))
)

final_fit = model_data.isel(
    time=np.flatnonzero(final_fit_mask.values)
)

print(f"Selected features: {len(FINAL_FEATURES)}")
print(f"Fitting issue dates: {final_fit.sizes['time']:,}")
print(
    "Period:",
    pd.Timestamp(final_fit.time.values[0]).date(),
    "to",
    pd.Timestamp(final_fit.time.values[-1]).date(),
)

Selected features: 10
Fitting issue dates: 1,089
Period: 2022-01-01 to 2024-12-24


In [21]:
def make_samples(dataset, horizon, predictors=None):
    predictors = feature_names if predictors is None else predictors

    target = dataset[f"chl_target_{horizon}d"].transpose(
        "time", "latitude", "longitude"
    )

    valid = (
        np.isfinite(target)
        & (target > 0)
        & (dataset["model_mask"] == 1)
    )

    positions = np.nonzero(valid.values)

    indexers = {
        dimension: xr.DataArray(position, dims="sample")
        for dimension, position in zip(target.dims, positions)
    }

    X = (
        dataset[predictors]
        .isel(indexers)
        .drop_vars(["time", "latitude", "longitude"])
        .to_dataframe()[predictors]
        .astype("float32")
    )

    return X, target.values[positions], positions


X_final_1d, chl_final_1d, final_positions_1d = make_samples(
    final_fit, 1, predictors=FINAL_FEATURES
)

print(f"Final +1-day training table: {X_final_1d.shape}")
display(X_final_1d.head())

Final +1-day training table: (7900523, 10)


,season_sin,season_cos,chl_latest,chl_age_days,chl_mean_7d,chl_observations_7d,latitude_deg,longitude_deg,sst_c,sst_observed
sample,,,,,,,,,,
0,0.0,1.0,NaN,NaN,NaN,0.0,35.554432,13.907421,NaN,0.0
1,0.0,1.0,NaN,NaN,NaN,0.0,35.554432,13.920269,NaN,0.0
2,0.0,1.0,NaN,NaN,NaN,0.0,35.554432,13.933117,NaN,0.0
3,0.0,1.0,NaN,NaN,NaN,0.0,35.554432,13.945964,NaN,0.0
4,0.0,1.0,NaN,NaN,NaN,0.0,35.554432,13.958812,NaN,0.0


In [22]:
final_models = {
    1: HistGradientBoostingRegressor(**FINAL_MODEL_PARAMS)
}

started = perf_counter()

final_models[1].fit(
    X_final_1d,
    np.log10(chl_final_1d).astype("float32"),
)

print(f"Final +1-day fit completed in {(perf_counter() - started) / 60:.1f} minutes")

# Release the large training arrays before preparing the next horizon.
del X_final_1d, chl_final_1d, final_positions_1d

Final +1-day fit completed in 0.5 minutes


In [23]:
X_final_4d, chl_final_4d, final_positions_4d = make_samples(
    final_fit, 4, predictors=FINAL_FEATURES
)

final_models[4] = HistGradientBoostingRegressor(**FINAL_MODEL_PARAMS)

started = perf_counter()

final_models[4].fit(
    X_final_4d,
    np.log10(chl_final_4d).astype("float32"),
)

print(f"Training rows: {len(chl_final_4d):,}")
print(f"Final +4-day fit completed in {(perf_counter() - started) / 60:.1f} minutes")

del X_final_4d, chl_final_4d, final_positions_4d

Training rows: 7,905,960
Final +4-day fit completed in 0.4 minutes


In [24]:
X_final_7d, chl_final_7d, final_positions_7d = make_samples(
    final_fit, 7, predictors=FINAL_FEATURES
)

final_models[7] = HistGradientBoostingRegressor(**FINAL_MODEL_PARAMS)

started = perf_counter()

final_models[7].fit(
    X_final_7d,
    np.log10(chl_final_7d).astype("float32"),
)

print(f"Training rows: {len(chl_final_7d):,}")
print(f"Final +7-day fit completed in {(perf_counter() - started) / 60:.1f} minutes")

del X_final_7d, chl_final_7d, final_positions_7d

Training rows: 7,922,162
Final +7-day fit completed in 0.5 minutes


In [25]:
with xr.open_dataset(
    BASE / "data/processed/malta_daily_observations.nc"
) as observations:
    final_training_chl = (
        observations["chl"]
        .sel(time=slice("2022-01-01", "2024-12-31"))
        .where(model_data["model_mask"] == 1)
        .load()
    )

final_monthly_climatology = (
    final_training_chl.groupby("time.month")
    .median(dim="time", skipna=True)
    .fillna(final_training_chl.median(dim="time", skipna=True))
)

test = model_data.isel(
    time=np.flatnonzero(model_data["split_test"].values == 1)
)

print(f"Final model horizons: {sorted(final_models)}")
print(f"Test issue dates: {test.sizes['time']}")
display(final_monthly_climatology)

Final model horizons: [1, 4, 7]
Test issue dates: 358


<xarray.DataArray 'chl' (month: 12, latitude: 133, longitude: 101)> Size: 645kB
array([[[0.09540617, 0.10286251, 0.0963295 , ..., 0.13389818,
         0.1291716 , 0.12725689],
        [0.10135216, 0.1001258 , 0.09253099, ..., 0.13301301,
         0.12820286, 0.13183467],
        [0.10273854, 0.09962878, 0.0966889 , ..., 0.13113672,
         0.12604545, 0.12668574],
        ...,
        [0.15878123, 0.16272138, 0.16428508, ..., 0.18556227,
         0.1668143 , 0.16382404],
        [0.16700593, 0.15867612, 0.16500849, ..., 0.16393757,
         0.15317175, 0.15955211],
        [0.16156296, 0.15837173, 0.15748534, ..., 0.16312355,
         0.16140689, 0.16029724]],

       [[0.10780803, 0.10642444, 0.10624504, ..., 0.13820444,
         0.14280932, 0.13863403],
        [0.10932746, 0.10594115, 0.10463269, ..., 0.13240086,
         0.1309678 , 0.1369771 ],
        [0.10999453, 0.10495982, 0.1047777 , ..., 0.13299146,
         0.13020691, 0.13103522],
...
        [0.06134211, 0.06118268, 0.06194308, ..., 0.10718367,
         0.0930189 , 0.08726785],
        [0.06224816, 0.06173193, 0.06319901, ..., 0.11174812,
         0.10263681, 0.09309077],
        [0.06355416, 0.06293381, 0.06151689, ..., 0.12413275,
         0.10944387, 0.10051224]],

       [[0.06560966, 0.06452348, 0.06390857, ..., 0.09688823,
         0.09936768, 0.09330875],
        [0.06753466, 0.06396703, 0.06346257, ..., 0.09443565,
         0.09773738, 0.09388829],
        [0.06316829, 0.06295969, 0.06394923, ..., 0.09883261,
         0.09705214, 0.09652197],
        ...,
        [0.11393557, 0.11096919, 0.11679503, ..., 0.12544306,
         0.1244104 , 0.12907478],
        [0.11391334, 0.11694147, 0.11799477, ..., 0.12623543,
         0.12629408, 0.12881416],
        [0.11608589, 0.12024336, 0.11549117, ..., 0.12358178,
         0.12840891, 0.13136758]]], shape=(12, 133, 101), dtype=float32)
Coordinates:
  * month      (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
  * latitude   (latitude) float32 532B 35.55 35.56 35.57 ... 36.87 36.88 36.89
  * longitude  (longitude) float32 404B 13.91 13.92 13.93 ... 15.17 15.18 15.19

In [26]:
X_test_1d, chl_test_1d, test_positions_1d = make_samples(
    test, 1, predictors=FINAL_FEATURES
)

prediction_final_1d = 10.0 ** final_models[1].predict(X_test_1d)

target_month = (test.time + np.timedelta64(1, "D")).dt.month

seasonal_test_grid = final_monthly_climatology.sel(
    month=target_month
).transpose("time", "latitude", "longitude")

test_predictions_1d = {
    "Persistence": X_test_1d["chl_latest"].to_numpy(),
    "Seasonal median": seasonal_test_grid.values[test_positions_1d],
    "Forecast model": prediction_final_1d,
}

common_test_1d = np.ones(len(chl_test_1d), dtype=bool)

for predicted in test_predictions_1d.values():
    common_test_1d &= np.isfinite(predicted) & (predicted > 0)

test_comparison_1d = pd.DataFrame([
    {
        "Method": method,
        **chl_scores(
            chl_test_1d[common_test_1d],
            predicted[common_test_1d],
        ),
    }
    for method, predicted in test_predictions_1d.items()
]).set_index("Method")

print(
    f"Scored targets: {common_test_1d.sum():,} / "
    f"{len(chl_test_1d):,}"
)

display(test_comparison_1d.style.format({
    "Scored pixel-days": "{:,.0f}",
    "MAE (mg/m³)": "{:.4f}",
    "RMSE (mg/m³)": "{:.4f}",
    "RMSE (log10)": "{:.4f}",
    "Median relative error (%)": "{:.1f}",
}))

Scored targets: 2,837,210 / 2,837,210


,Scored pixel-days,MAE (mg/m³),RMSE (mg/m³),RMSE (log10),Median relative error (%)
Method,,,,,
Persistence,"2,837,210",0.0136,0.0404,0.1075,12.7
Seasonal median,"2,837,210",0.0164,0.0389,0.1244,15.1
Forecast model,"2,837,210",0.0121,0.0341,0.0963,11.0


In [27]:
X_test_4d, chl_test_4d, test_positions_4d = make_samples(
    test, 4, predictors=FINAL_FEATURES
)

prediction_final_4d = 10.0 ** final_models[4].predict(X_test_4d)

target_month = (test.time + np.timedelta64(4, "D")).dt.month

seasonal_test_grid_4d = final_monthly_climatology.sel(
    month=target_month
).transpose("time", "latitude", "longitude")

test_predictions_4d = {
    "Persistence": X_test_4d["chl_latest"].to_numpy(),
    "Seasonal median": seasonal_test_grid_4d.values[test_positions_4d],
    "Forecast model": prediction_final_4d,
}

common_test_4d = np.ones(len(chl_test_4d), dtype=bool)

for predicted in test_predictions_4d.values():
    common_test_4d &= np.isfinite(predicted) & (predicted > 0)

test_comparison_4d = pd.DataFrame([
    {
        "Method": method,
        **chl_scores(
            chl_test_4d[common_test_4d],
            predicted[common_test_4d],
        ),
    }
    for method, predicted in test_predictions_4d.items()
]).set_index("Method")

print(f"Scored targets: {common_test_4d.sum():,} / {len(chl_test_4d):,}")

display(test_comparison_4d.style.format({
    "Scored pixel-days": "{:,.0f}",
    "MAE (mg/m³)": "{:.4f}",
    "RMSE (mg/m³)": "{:.4f}",
    "RMSE (log10)": "{:.4f}",
    "Median relative error (%)": "{:.1f}",
}))

Scored targets: 2,846,879 / 2,846,879


,Scored pixel-days,MAE (mg/m³),RMSE (mg/m³),RMSE (log10),Median relative error (%)
Method,,,,,
Persistence,"2,846,879",0.0155,0.0449,0.1230,14.3
Seasonal median,"2,846,879",0.0163,0.0385,0.1237,15.0
Forecast model,"2,846,879",0.0134,0.0356,0.1065,12.4


In [28]:
X_test_7d, chl_test_7d, test_positions_7d = make_samples(
    test, 7, predictors=FINAL_FEATURES
)

prediction_final_7d = 10.0 ** final_models[7].predict(X_test_7d)

target_month = (test.time + np.timedelta64(7, "D")).dt.month

seasonal_test_grid_7d = final_monthly_climatology.sel(
    month=target_month
).transpose("time", "latitude", "longitude")

test_predictions_7d = {
    "Persistence": X_test_7d["chl_latest"].to_numpy(),
    "Seasonal median": seasonal_test_grid_7d.values[test_positions_7d],
    "Forecast model": prediction_final_7d,
}

common_test_7d = np.ones(len(chl_test_7d), dtype=bool)

for predicted in test_predictions_7d.values():
    common_test_7d &= np.isfinite(predicted) & (predicted > 0)

test_comparison_7d = pd.DataFrame([
    {
        "Method": method,
        **chl_scores(
            chl_test_7d[common_test_7d],
            predicted[common_test_7d],
        ),
    }
    for method, predicted in test_predictions_7d.items()
]).set_index("Method")

print(f"Scored targets: {common_test_7d.sum():,} / {len(chl_test_7d):,}")

display(test_comparison_7d.style.format({
    "Scored pixel-days": "{:,.0f}",
    "MAE (mg/m³)": "{:.4f}",
    "RMSE (mg/m³)": "{:.4f}",
    "RMSE (log10)": "{:.4f}",
    "Median relative error (%)": "{:.1f}",
}))

Scored targets: 2,852,080 / 2,852,080


,Scored pixel-days,MAE (mg/m³),RMSE (mg/m³),RMSE (log10),Median relative error (%)
Method,,,,,
Persistence,"2,852,080",0.0169,0.0457,0.1343,16.1
Seasonal median,"2,852,080",0.0162,0.0384,0.1235,15.0
Forecast model,"2,852,080",0.0143,0.0366,0.1135,13.6


In [29]:
test_results = pd.concat(
    [test_comparison_1d, test_comparison_4d, test_comparison_7d],
    keys=["+1d", "+4d", "+7d"],
    names=["Horizon", "Method"],
)

final_summary = test_results.xs(
    "Forecast model", level="Method"
).copy()

persistence_scores = test_results.xs(
    "Persistence", level="Method"
)

final_summary["RMSE reduction vs persistence (%)"] = (
    100 * (
        1 - final_summary["RMSE (mg/m³)"]
        / persistence_scores["RMSE (mg/m³)"]
    )
)

display(final_summary.style.format({
    "Scored pixel-days": "{:,.0f}",
    "MAE (mg/m³)": "{:.4f}",
    "RMSE (mg/m³)": "{:.4f}",
    "RMSE (log10)": "{:.4f}",
    "Median relative error (%)": "{:.1f}",
    "RMSE reduction vs persistence (%)": "{:.1f}",
}))

,Scored pixel-days,MAE (mg/m³),RMSE (mg/m³),RMSE (log10),Median relative error (%),RMSE reduction vs persistence (%)
Horizon,,,,,,
+1d,"2,837,210",0.0121,0.0341,0.0963,11.0,15.7
+4d,"2,846,879",0.0134,0.0356,0.1065,12.4,20.7
+7d,"2,852,080",0.0143,0.0366,0.1135,13.6,19.7


In [30]:
import joblib
import sklearn
from datetime import datetime, timezone

model_dir = BASE / "models"
model_dir.mkdir(parents=True, exist_ok=True)

bundle_path = model_dir / "chl_forecast_1_4_7d.joblib"

model_bundle = {
    "models": final_models,
    "feature_names": FINAL_FEATURES.copy(),
    "forecast_horizons_days": list(horizons),
    "forecast_issue_time_utc": model_data.attrs["forecast_issue_time_utc"],
    "target_transform": "log10",
    "target_units": "milligram m^-3",
    "preprocessing": {
        "chl_availability_lag_days": 1,
        "sst_availability_lag_days": 0,
        "chl_history_window_days": 7,
        "sst_units": "degree_Celsius",
    },
    "training_issue_date_range": [
        pd.Timestamp(final_fit.time.values[0]).date().isoformat(),
        pd.Timestamp(final_fit.time.values[-1]).date().isoformat(),
    ],
    "availability_basis": model_data.attrs["availability_basis"],
    "model_mask": model_data["model_mask"].load().copy(deep=True),
    "monthly_climatology": final_monthly_climatology.load().copy(deep=True),
    "test_year": 2025,
    "test_results": test_results.copy(),
    "sklearn_version": sklearn.__version__,
    "created_utc": datetime.now(timezone.utc).isoformat(),
}

joblib.dump(model_bundle, bundle_path, compress=3)

print(f"Saved: {bundle_path}")
print(f"File size: {bundle_path.stat().st_size / 1024**2:,.2f} MB")

Saved: models\chl_forecast_1_4_7d.joblib
File size: 1.20 MB


In [31]:
loaded_bundle = joblib.load(bundle_path)

assert loaded_bundle["feature_names"] == FINAL_FEATURES
assert set(loaded_bundle["models"]) == set(horizons)

sample_inputs = X_test_1d[FINAL_FEATURES].iloc[:100]

for horizon in horizons:
    original = final_models[horizon].predict(sample_inputs)
    reloaded = loaded_bundle["models"][horizon].predict(sample_inputs)

    np.testing.assert_allclose(
        reloaded, original, rtol=1e-12, atol=1e-12
    )

    print(f"+{horizon}-day model: reload verified")

print(f"Saved feature count: {len(loaded_bundle['feature_names'])}")

+1-day model: reload verified
+4-day model: reload verified
+7-day model: reload verified
Saved feature count: 10
